In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Carrega os históricos mensais
df_transactions = spark.table("churn_lakehouse.silver.stg_transactions")
df_financial    = spark.table("churn_lakehouse.silver.stg_financial")
df_customers    = spark.table("churn_lakehouse.silver.stg_crm")

print(f"Transações:  {df_transactions.count():,} linhas")
print(f"Financeiro:  {df_financial.count():,} linhas")
print(f"Clientes:    {df_customers.count():,} linhas")

Transações:  46,657 linhas
Financeiro:  46,657 linhas
Clientes:    2,000 linhas


In [0]:
w_customer = Window.partitionBy("customer_id")

df_pay_features = df_transactions.withColumn(
    "is_recent",
    F.when(
        F.col("invoice_date") >= F.date_sub(F.current_date(), 90),
        1
    ).otherwise(0)
).groupBy("customer_id").agg(

    # Total de faturas recentes e antigas
    F.sum(F.col("is_recent")).alias("invoices_last_3m"),
    F.sum(F.when(F.col("is_recent") == 0, 1).otherwise(0)).alias("invoices_before_3m"),

    # Atrasos recentes e antigos
    F.sum(F.when((F.col("is_recent") == 1) & F.col("is_late"), 1).otherwise(0)).alias("late_last_3m"),
    F.sum(F.when((F.col("is_recent") == 0) & F.col("is_late"), 1).otherwise(0)).alias("late_before_3m"),

    # Valor não pago recente
    F.round(
        F.sum(F.when(F.col("is_recent") == 1, F.col("amount_unpaid_brl")).otherwise(0))
    , 2).alias("unpaid_last_3m"),

    # Máximo de dias em atraso
    F.max(F.col("days_late")).alias("max_days_late_ever"),

    # Volatilidade de pagamento
    F.round(F.stddev("days_late"), 2).alias("stddev_days_late"),

    # Descontos de retenção recentes
    F.sum(
        F.when((F.col("is_recent") == 1) & (F.col("discount_pct") > 0), 1).otherwise(0)
    ).alias("retention_discounts_last_3m"),

).withColumn(
    # Taxa de atraso recente — com proteção contra divisão por zero
    "late_rate_last_3m",
    F.when(
        F.col("invoices_last_3m") > 0,
        F.round(F.col("late_last_3m") / F.col("invoices_last_3m"), 4)
    ).otherwise(0.0)
).withColumn(
    # Taxa de atraso anterior
    "late_rate_before_3m",
    F.when(
        F.col("invoices_before_3m") > 0,
        F.round(F.col("late_before_3m") / F.col("invoices_before_3m"), 4)
    ).otherwise(0.0)
).withColumn(
    # Delta de tendência — positivo significa piorando
    "late_rate_delta",
    F.round(F.col("late_rate_last_3m") - F.col("late_rate_before_3m"), 4)
).drop("invoices_last_3m", "invoices_before_3m", "late_last_3m", "late_before_3m")

print(f"Features de pagamento: {df_pay_features.count():,} clientes")
df_pay_features.show(3)

Features de pagamento: 2,000 clientes
+-----------+--------------+------------------+----------------+---------------------------+-----------------+-------------------+---------------+
|customer_id|unpaid_last_3m|max_days_late_ever|stddev_days_late|retention_discounts_last_3m|late_rate_last_3m|late_rate_before_3m|late_rate_delta|
+-----------+--------------+------------------+----------------+---------------------------+-----------------+-------------------+---------------+
| CUST-01108|           0.0|                36|            8.28|                          0|              0.0|              0.087|         -0.087|
| CUST-01029|         189.9|                40|           10.51|                          1|              0.0|             0.1837|        -0.1837|
| CUST-01113|           0.0|                24|            4.44|                          0|              0.0|             0.0816|        -0.0816|
+-----------+--------------+------------------+----------------+----------------

In [0]:
# Ordena por mês para calcular tendências
w_fin = Window.partitionBy("customer_id").orderBy("month")

df_fin_features = df_financial.withColumn(
    "row_num", F.row_number().over(w_fin)
).withColumn(
    "total_months", F.count("customer_id").over(Window.partitionBy("customer_id"))
).withColumn(
    "is_recent_fin",
    F.when(F.col("row_num") >= F.col("total_months") - 2, 1).otherwise(0)
).groupBy("customer_id").agg(

    # MRR médio recente vs histórico — captura downgrade
    F.round(
        F.avg(F.when(F.col("is_recent_fin") == 1, F.col("mrr_brl"))),
    2).alias("mrr_avg_last_3m"),

    F.round(
        F.avg(F.when(F.col("is_recent_fin") == 0, F.col("mrr_brl"))),
    2).alias("mrr_avg_before_3m"),

    # Expansão e contração recentes
    F.round(
        F.sum(F.when(F.col("is_recent_fin") == 1, F.col("expansion_brl")).otherwise(0))
    , 2).alias("expansion_last_3m"),

    F.round(
        F.sum(F.when(F.col("is_recent_fin") == 1, F.col("contraction_brl")).otherwise(0))
    , 2).alias("contraction_last_3m"),

    # Volatilidade do MRR
    F.round(F.stddev("mrr_brl"), 2).alias("mrr_stddev"),

    # Quantos meses com contração
    F.sum(
        F.when(F.col("contraction_brl") > 0, 1).otherwise(0)
    ).alias("months_with_contraction"),

    # Quantos meses com expansão
    F.sum(
        F.when(F.col("expansion_brl") > 0, 1).otherwise(0)
    ).alias("months_with_expansion"),

).withColumn(
    # Delta de MRR — negativo significa downgrade
    "mrr_trend",
    F.round(
        F.col("mrr_avg_last_3m") - F.col("mrr_avg_before_3m")
    , 2)
).withColumn(
    # Taxa de contração — proporção de meses com queda
    "contraction_rate",
    F.round(
        F.col("months_with_contraction") /
        (F.col("months_with_contraction") + F.col("months_with_expansion") + 0.001)
    , 4)
)

print(f"Features financeiras: {df_fin_features.count():,} clientes")
df_fin_features.show(3)

Features financeiras: 2,000 clientes
+-----------+---------------+-----------------+-----------------+-------------------+----------+-----------------------+---------------------+---------+----------------+
|customer_id|mrr_avg_last_3m|mrr_avg_before_3m|expansion_last_3m|contraction_last_3m|mrr_stddev|months_with_contraction|months_with_expansion|mrr_trend|contraction_rate|
+-----------+---------------+-----------------+-----------------+-------------------+----------+-----------------------+---------------------+---------+----------------+
| CUST-00001|          499.0|            899.0|              0.0|              100.0|    116.51|                      1|                    0|   -400.0|           0.999|
| CUST-00002|          899.0|            899.0|            450.0|                0.0|       0.0|                      0|                    3|      0.0|             0.0|
| CUST-00003|          899.0|            899.0|            300.0|                0.0|       0.0|                 

In [0]:
# Carrega a feature store atual como base
df_base = spark.table("churn_lakehouse.gold.gold_feature_store")

# Une com as novas features temporais
df_enhanced = df_base \
    .join(df_pay_features, on="customer_id", how="left") \
    .join(df_fin_features, on="customer_id", how="left")

# Preenche nulos com 0 nas novas features
new_cols = [c for c in df_enhanced.columns if c not in df_base.columns]
for col in new_cols:
    df_enhanced = df_enhanced.withColumn(col, F.coalesce(F.col(col), F.lit(0.0)))

print(f"Features originais:  {len(df_base.columns)}")
print(f"Features novas:      {len(new_cols)}")
print(f"Features totais:     {len(df_enhanced.columns)}")
print(f"Clientes:            {df_enhanced.count():,}")
print(f"\nNovas features adicionadas:\n{new_cols}")

# Grava como nova versão da feature store
(
    df_enhanced
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("churn_lakehouse.gold.gold_feature_store_v2")
)

print("\n✓ gold_feature_store_v2 gravada com sucesso")

Features originais:  41
Features novas:      16
Features totais:     57
Clientes:            2,000

Novas features adicionadas:
['unpaid_last_3m', 'max_days_late_ever', 'stddev_days_late', 'retention_discounts_last_3m', 'late_rate_last_3m', 'late_rate_before_3m', 'late_rate_delta', 'mrr_avg_last_3m', 'mrr_avg_before_3m', 'expansion_last_3m', 'contraction_last_3m', 'mrr_stddev', 'months_with_contraction', 'months_with_expansion', 'mrr_trend', 'contraction_rate']

✓ gold_feature_store_v2 gravada com sucesso
